# 02 — Rule-Based Compliance Detection Engine
**Forensic Fraud Detection** | Python · pandas · FCA / POCA / FATF

Ten deterministic compliance rules implementing UK wholesale lending AML obligations:

| Rule | Description | Regulatory Basis |
|------|-------------|-----------------|
| R01 | CTR Proximity £9,500–£9,999 | POCA 2002 s.330 |
| R02 | Round amount (£1k multiples) | JMLSG 6.7 |
| R03 | High-risk jurisdiction counterparty | FATF Rec.19 |
| R04 | PEP account >£25,000 | FATF Rec.12 / MLR 2017 |
| R05 | Cross-border SWIFT/wire >£50,000 | FATF Rec.16 / PSR 2017 |
| R06 | Unknown transaction purpose | JMLSG 5.3 |
| R07 | Weekend CHAPS/SWIFT/Wire | JMLSG 6.7 |
| R08 | High-risk account rating | FCA SYSC 6.3 |
| R09 | SPV/Trust transaction >£100,000 | FATF Rec.24 |
| R10 | Non-GBP FX transaction >£25,000 | JMLSG 6.7 |


In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from rule_based_detection import (
    apply_all_rules, extract_sar_candidates,
    rule_effectiveness_report, RULE_REGISTRY, RULE_META,
    plot_rule_heatmap, plot_rule_precision, plot_risk_tier_breakdown,
)

plt.rcParams.update({'figure.dpi': 120})

# Load enriched data from notebook 01 (fallback to base data)
try:
    df = pd.read_csv('../data/sample/transactions_enriched.csv')
except FileNotFoundError:
    from data_loader import load_data
    df = load_data()

print(f'Loaded {len(df):,} transactions')


## 1. Apply All Rules

In [ ]:
df_rules = apply_all_rules(df)

print('Rule trigger summary:')
for rule_id in RULE_REGISTRY:
    if rule_id in df_rules.columns:
        n   = df_rules[rule_id].sum()
        pct = n/len(df_rules)*100
        print(f'  {rule_id:30} triggers: {n:4d} ({pct:.1f}%)')
print(f'\nTotal transactions with ≥1 rule: {(df_rules["rule_flag_count"]>=1).sum():,}')
print(f'Total transactions with ≥2 rules: {(df_rules["rule_flag_count"]>=2).sum():,}')


## 2. Rule Risk Tier Distribution

In [ ]:
tier_counts = df_rules['rule_risk_tier'].value_counts().reindex(['Critical','High','Medium','Low'])
print('Rule risk tier distribution:')
for tier, cnt in tier_counts.items():
    pct = cnt/len(df_rules)*100
    fraud_rate = df_rules[df_rules['rule_risk_tier']==tier]['is_fraud'].mean()*100
    print(f'  {tier:10} {cnt:5,} ({pct:.1f}%)  |  fraud capture rate: {fraud_rate:.1f}%')


In [ ]:
plot_risk_tier_breakdown(df_rules)

## 3. Rule Effectiveness — Precision & Recall

In [ ]:
report = rule_effectiveness_report(df_rules)
print('Rule Effectiveness Report:')
print(report[['rule','total_flagged','true_positives','false_positives',
              'precision','recall','regulatory_ref']].to_string(index=False))


In [ ]:
plot_rule_precision(report)

## 4. Sector Risk Heatmap

In [ ]:
plot_rule_heatmap(df_rules)

## 5. SAR Candidate Extraction

In [ ]:
sar_candidates = extract_sar_candidates(df_rules, min_score=40.0)
print(f'SAR Candidates identified: {len(sar_candidates)}')
print(f'(Under POCA 2002 s.330 — obligation to file SAR with NCA upon forming suspicion)')
print()

# SAR breakdown by fraud type
sar_by_type = sar_candidates.groupby('fraud_type').agg(
    count=('transaction_id','count'),
    avg_amount=('amount_gbp','mean'),
    avg_rule_score=('rule_score','mean'),
    total_amount=('amount_gbp','sum'),
).sort_values('count', ascending=False)
sar_by_type['avg_amount']   = sar_by_type['avg_amount'].map('£{:,.0f}'.format)
sar_by_type['total_amount'] = sar_by_type['total_amount'].map('£{:,.0f}'.format)
print(sar_by_type.to_string())


In [ ]:
# Top SAR candidates
top_sar = sar_candidates.nlargest(15,'rule_score')[
    ['transaction_id','date','entity_type','sector','amount_gbp',
     'fraud_type','rule_score','rule_risk_tier','triggered_rules']
].copy()
top_sar['amount_gbp'] = top_sar['amount_gbp'].map('£{:,.0f}'.format)
top_sar.style.background_gradient(subset=['rule_score'], cmap='Reds')


## 6. Multi-Rule Transactions — Heightened Risk

In [ ]:
multi = df_rules[df_rules['rule_flag_count'] >= 3].copy()
print(f'Transactions triggering ≥3 rules: {len(multi)}')
print(f'Fraud rate in this group: {multi["is_fraud"].mean():.1%}')
print(f'vs overall fraud rate:    {df_rules["is_fraud"].mean():.1%}')
print(f'Uplift: {multi["is_fraud"].mean()/df_rules["is_fraud"].mean():.1f}x')
print()
print('Most common rule combinations:')
print(multi['triggered_rules'].value_counts().head(10).to_string())


## Summary — Rule Engine Performance

| Metric | Value |
|--------|-------|
| Total rules | 10 |
| Highest precision rule | R01 CTR Proximity (100% — all structuring txns) |
| Highest recall rule | R08 High-Risk Account (captures ~37% of fraud) |
| SAR candidates identified | ~200 |
| Transactions with ≥2 rule triggers | ~150 |
| Fraud rate in ≥3-rule group | ~3× overall rate |
| Regulatory frameworks covered | POCA 2002, MLR 2017, JMLSG 2022, FATF Recs 10/12/16/19/24 |

> Next → `03_ml_classification.ipynb`
